#ingest circuit.csv file
1.read file using spark dataframe reader API
2.Add metadata columns
    . source file
    . ingestion time stamp
3.write to bronze delta table



In [0]:
%run ../00.common/1.environment_config

In [0]:
%run ../00.common/2.bronze_helper

In [0]:
source_file=f"{landing_path}/circuits.csv"
table_name=f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType,StructField,DoubleType,StringType
circuit_schema=StructType([StructField('circuitid',StringType(),True),
                       StructField('url',StringType(),True),
                       StructField('circuitName',StringType(),True),
                       StructField('lat',DoubleType(),True),
                       StructField('long',DoubleType(),True),
                       StructField('locality',StringType(),True),
                       StructField('country',StringType(),True)

])


In [0]:

circuits_df=spark.read.format('csv').option('header', 'true').schema(circuit_schema).load(source_file)


In [0]:
circuits_final_df=add_ingestion_metadata(circuits_df)

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%sql
select * from formula1.bronze.circuits;

In [0]:
display(spark.table(table_name))